In [50]:
import pandas as pd
df = pd.read_csv('ncr_ride_bookings.csv')


In [51]:
df.head()

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [52]:
# Create a summary table
summary = pd.DataFrame({
    'Non-Null': df.count(),
    'Null': df.isnull().sum()
})

print(summary)

                                   Non-Null    Null
Date                                 150000       0
Time                                 150000       0
Booking ID                           150000       0
Booking Status                       150000       0
Customer ID                          150000       0
Vehicle Type                         150000       0
Pickup Location                      150000       0
Drop Location                        150000       0
Avg VTAT                             139500   10500
Avg CTAT                             102000   48000
Cancelled Rides by Customer           10500  139500
Reason for cancelling by Customer     10500  139500
Cancelled Rides by Driver             27000  123000
Driver Cancellation Reason            27000  123000
Incomplete Rides                       9000  141000
Incomplete Rides Reason                9000  141000
Booking Value                        102000   48000
Ride Distance                        102000   48000
Driver Ratin

In [53]:
df['Booking ID'] = df['Booking ID'].str.strip('"')
df['Customer ID'] = df['Customer ID'].str.strip('"')

In [54]:
df = df.drop_duplicates(subset=['Booking ID'], keep='first')
print(f"Rows after dropping duplicates: {len(df)}")

Rows after dropping duplicates: 148767


In [55]:
mapping = {
    'Completed'            : 'Successful',
    'No Driver Found'      : 'Supply Gap',
    'Incomplete'           : 'Incomplete Ride',
    'Cancelled by Driver'  : 'Driver Cancellation',
    'Cancelled by Customer': 'Customer Cancellation'
}
df['Status_Group'] = df['Booking Status'].map(mapping)

# Verify — should be 0
print("Unmapped Status_Group rows:", df['Status_Group'].isnull().sum())

Unmapped Status_Group rows: 0


In [56]:
df['Cancelled Rides by Customer'] = df['Cancelled Rides by Customer'].fillna(0)
df['Cancelled Rides by Driver']   = df['Cancelled Rides by Driver'].fillna(0)
df['Incomplete Rides']            = df['Incomplete Rides'].fillna(0)

In [57]:
reason_cols = [
    'Reason for cancelling by Customer',
    'Driver Cancellation Reason',
    'Incomplete Rides Reason'
]
df[reason_cols] = df[reason_cols].fillna('Not Applicable')

In [58]:
print("Driver Cancellation nulls:", df['Driver Cancellation Reason'].isnull().sum())


Driver Cancellation nulls: 0


In [59]:
df['Avg VTAT'] = df['Avg VTAT'].fillna(0)


In [60]:
df['Avg CTAT'] = df['Avg CTAT'].fillna(0)

In [61]:
print("Avg CTAT dtype (must be float64):", df['Avg CTAT'].dtype)

Avg CTAT dtype (must be float64): float64


In [62]:
completed_mask = df['Booking Status'] == 'Completed'
valid_ratings   = df.loc[completed_mask, 'Driver Ratings'].dropna()
mean_rating     = valid_ratings.mean()
print(f"Mean Driver Rating (completed rides only): {mean_rating:.4f}")

Mean Driver Rating (completed rides only): 4.2308


In [63]:
fill_mask = completed_mask & df['Driver Ratings'].isnull()
print(f"Completed rides missing a rating: {fill_mask.sum()}")
df.loc[fill_mask, 'Driver Ratings'] = round(mean_rating, 2)

Completed rides missing a rating: 0


In [64]:
non_completed_ratings = df.loc[~completed_mask, 'Driver Ratings'].notnull().sum()
print(f"Non-completed rides with ratings (should be 0): {non_completed_ratings}")


Non-completed rides with ratings (should be 0): 0


In [65]:
valid_cust_ratings = df.loc[completed_mask, 'Customer Rating'].dropna()
mean_cust_rating   = valid_cust_ratings.mean()
print(f"Mean Customer Rating (completed rides only): {mean_cust_rating:.4f}")

fill_mask_cust = completed_mask & df['Customer Rating'].isnull()
df.loc[fill_mask_cust, 'Customer Rating'] = round(mean_cust_rating, 2)

Mean Customer Rating (completed rides only): 4.4043


In [66]:
completed_missing_value = df[completed_mask & df['Booking Value'].isnull()]
print(f"Completed rides missing Booking Value (should be 0): {len(completed_missing_value)}")

Completed rides missing Booking Value (should be 0): 0


In [67]:
completed_missing_dist = df[completed_mask & df['Ride Distance'].isnull()]
print(f"Completed rides missing Ride Distance (should be 0): {len(completed_missing_dist)}")

Completed rides missing Ride Distance (should be 0): 0


In [68]:
transaction_mask = df['Booking Status'].isin(['Completed', 'Incomplete'])
missing_payment_in_transactions = df[transaction_mask & df['Payment Method'].isnull()]
print(f"Transactions missing Payment Method: {len(missing_payment_in_transactions)}")

Transactions missing Payment Method: 0


In [69]:
df['Payment Method'] = df['Payment Method'].str.strip().str.title()

In [70]:
print("\n=== FINAL NULL SUMMARY ===")
summary = pd.DataFrame({
    'Non-Null Count': df.count(),
    'Null Count'    : df.isnull().sum()
})
print(summary)

print("\n=== DTYPE CHECK — no object columns that should be numeric ===")
print(df[['Avg VTAT', 'Avg CTAT', 'Driver Ratings', 'Customer Rating',
          'Booking Value', 'Ride Distance']].dtypes)

print("\n=== STATUS GROUP DISTRIBUTION ===")
print(df['Status_Group'].value_counts())


=== FINAL NULL SUMMARY ===
                                   Non-Null Count  Null Count
Date                                       148767           0
Time                                       148767           0
Booking ID                                 148767           0
Booking Status                             148767           0
Customer ID                                148767           0
Vehicle Type                               148767           0
Pickup Location                            148767           0
Drop Location                              148767           0
Avg VTAT                                   148767           0
Avg CTAT                                   148767           0
Cancelled Rides by Customer                148767           0
Reason for cancelling by Customer          148767           0
Cancelled Rides by Driver                  148767           0
Driver Cancellation Reason                 148767           0
Incomplete Rides                          

In [71]:
%pip install pymysql cryptography
import pandas as pd 
from sqlalchemy import create_engine

# 1. Define your MySQL connection credentials (from your Workbench settings)
USER = 'root'
PASSWORD = 'root'        # Your MySQL Workbench password
HOST = '127.0.0.1'       # Localhost IP
PORT = '3308'            # Your specific port number
DATABASE = 'govt'        # Make sure this schema exists inside your Workbench

# 2. Create the connection engine
# This uses the pymysql library we just installed to talk to MySQL
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}")

# 3. Export your cleaned DataFrame to a MySQL table
table_name = 'cleaned_ride_bookings'

print("Exporting data to MySQL Workbench... please wait.")

df.to_sql(
    name=table_name, 
    con=engine, 
    if_exists='replace', # Overwrites the table if it already exists
    index=False          # Keeps your clean columns without adding a row index column
)

print(f"🎉 Success! Data successfully exported to the table '{table_name}' in database '{DATABASE}'!")

Note: you may need to restart the kernel to use updated packages.
Exporting data to MySQL Workbench... please wait.


You should consider upgrading via the 'c:\Users\HP\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


🎉 Success! Data successfully exported to the table 'cleaned_ride_bookings' in database 'govt'!
